# Percie fine-tune 4/5 — Images + video (Qwen2.5-VL-3B)

Fine-tunes **Qwen2.5-VL-3B-Instruct** on counting, reading text, general questions and short video clips, using the app's exact prompts and JSON formats.

**Data:**
- Images: HuggingFaceM4 Cauldron **train** subsets `tallyqa` (counting), `textvqa` + `st_vqa` (reading text), `vqav2` (general).
- Video: synthetic clips generated in this notebook with known answers: shapes appearing, moving, bouncing, signs changing, countdowns. The frames are sampled the same way the app samples them.

**Scored on:** held-out Cauldron images and unseen synthetic clips.

**Do this first:** run once with `SMOKE_TEST = True` (about 10 minutes). It trains 10 steps, exports, and loads the model in Ollama inside Colab. Only if that works, set it to `False` and run the real training (about 1–2 h on an A100).

**Keep it only if it beats stock `qwen3-vl:4b`** on the app's image and video eval.

⚠️ **Hackathon rule:** train only on TRAIN splits. Every practice item comes from a validation/test split, and the judges' items almost certainly do too. Never train on the practice pack.

## 1. Install

In [ ]:
!pip install -q unsloth
!pip install -q opencv-python-headless

## 2. ⚙️ Settings

In [ ]:
MODEL_NAME  = "unsloth/Qwen2.5-VL-3B-Instruct"
RUN_NAME    = "neuro-vision"
OLLAMA_NAME = "neuro-vision"
APP_SETTING = "NEUROASSIST_VISION_MODEL"

# FIRST RUN: keep True. Trains 10 steps, exports, and loads the result in Ollama inside Colab,
# proving the whole path works before you spend an hour training. Then set False and Run all.
SMOKE_TEST = True

# Image data: Cauldron TRAIN subsets (examples per subset)
CAULDRON = {"tallyqa": 2500,   # counting  (COCO train / Visual Genome images)
            "textvqa": 2000,   # reading text in photos
            "st_vqa": 1500,    # reading scene text
            "vqav2": 2000}     # general questions (COCO train2014 images)
# Video data: synthetic clips generated in this notebook (shapes, motion, signs, counting)
SYNTH_VIDEOS = 3000
EVAL_IMAGES, EVAL_VIDEOS = 100, 100

# Must match the app's "standard" profile (neuroassist/config.py)
VIDEO_FPS_DENSE, VIDEO_FPS_SPARSE = 4.0, 1.0
VIDEO_MAX_FRAMES, VIDEO_SIDE = 16, 448
IMAGE_SIDE = 768                 # training images are capped here to save memory

MAX_SEQ_LENGTH = 6144            # 16 frames x ~256 tokens + text
LORA_RANK, LORA_ALPHA = 16, 16
EPOCHS = 1
LEARNING_RATE = 2e-4
BATCH_SIZE, GRAD_ACCUM = 2, 4
SEED = 3407
SAVE_TO_DRIVE, DRIVE_DIR = True, "/content/drive/MyDrive/neuroassist-finetunes"

# ---- Copied from the app at build time (do not edit; re-run finetune/build_notebooks.py) ----
IMAGE_PROMPTS = {'read_text': "Read the text in this image exactly as written. In 'answer' give only the main text; in 'detail' say where it appears.", 'identify': "{q}\nIn 'answer' name the main object in a few words (brand or type if visible). In 'detail' give one short sentence of description.", 'describe': "{q}\nIn 'answer' give a direct, short answer. In 'detail' give one or two sentences describing what is in the image.", 'summarize': "{q}\nIn 'answer' give the key point in one sentence. In 'detail' list the important things visible, separated by semicolons."}
COUNT_SUFFIX  = " Count carefully, including small, partly hidden or background ones."
VIDEO_HINTS   = {'count_events': 'Compare consecutive frames carefully and count each event once.', 'count': 'Things may appear one at a time; count every distinct one across all frames.', 'sequence': 'Pay close attention to the first and the last frames.', 'motion': 'Compare positions and sizes between the first and last frames.'}
ROUTER_TASKS  = [('write_code', '\\bdef \\w+\\(|complete (this|the) (python )?function|write (a |some )?(python )?(function|code|script)'), ('transcribe', '\\btranscri|word for word|what (is|are) (he|she|they|the speaker) saying'), ('identify_sound', 'what (sound|noise)|what (do|can) (you|i) hear|which (sound|noise)'), ('count_events', 'how many times|bounce|appear.*disappear|\\bleft at the end'), ('count', '\\bhow many\\b|\\bcount\\b|number of'), ('sequence', '\\bfirst\\b|\\blast\\b|\\bat the (start|end|beginning)\\b|\\bin the end\\b'), ('motion', 'direction|which way|\\bmov(e|es|ing)\\b|toward|taller|shorter|bigger|smaller|faster'), ('read_text', "what does (it|this|that|the \\w+) say|\\bread\\b|\\btitle\\b|\\bsign\\b|\\blabel\\b|what('s| is) written"), ('identify', "what('s| is) (this|that|it)\\b|what kind of|what type of|what is (this|the) (item|object|thing)|what is in"), ('summarize', 'summar|simplif|explain|key points|\\bnotes\\b|tl;?dr|break (it|this) down')]
ALLOWED_IMAGE, ALLOWED_VIDEO, DENSE_TASKS = set(['count', 'identify', 'read_text', 'summarize']), set(['count', 'count_events', 'motion', 'read_text', 'sequence', 'summarize']), set(['count', 'count_events', 'motion', 'sequence'])

## 3. App formats (prompts, routing, frame sampling)

In [ ]:
import io, json, random, re
import numpy as np, cv2
from PIL import Image

def app_task(question, allowed, default="describe"):
    """Same rules as the app's router (neuroassist/router.py)."""
    for task, pat in ROUTER_TASKS:
        if task in allowed and re.search(pat, question, re.I):
            return task
    return default

NUM_WORDS = {w: i for i, w in enumerate("zero one two three four five six seven eight nine ten eleven twelve".split())}
def to_int(s):
    s = s.strip().lower().rstrip(".")
    return int(s) if s.isdigit() else NUM_WORDS.get(s)

def image_example(question, answer):
    """(prompt, target) exactly as the app asks and parses (neuroassist/tools/image.py)."""
    task = app_task(question, ALLOWED_IMAGE)
    if task == "count":                     # trains the app's COUNT_MODE=direct path
        n = to_int(answer)
        if n is None:
            return None
        return question + COUNT_SUFFIX, json.dumps({"count": n, "detail": ""})
    prompt = IMAGE_PROMPTS.get(task, IMAGE_PROMPTS["describe"]).format(q=question)
    return prompt, json.dumps({"answer": answer, "detail": ""})

def video_prompt(question, times, duration, task):
    """Exactly the app's direct video prompt (neuroassist/tools/video.py)."""
    labels = "\n".join(f"Image {i + 1} is at t={t:.1f}s." for i, t in enumerate(times))
    return (f"These {len(times)} images are frames from a {duration:.1f}-second video, in time order.\n"
            f"{labels}\n{VIDEO_HINTS.get(task, '')}\n\nQuestion: {question}\n"
            "Answer in one short, direct sentence.")

def frame_changed(a, b, mean_thresh=1.5, local_thresh=30.0):
    """Same test as the app (neuroassist/ingest.py): whole-picture change OR a sharp local change."""
    ga, gb = (cv2.cvtColor(x, cv2.COLOR_BGR2GRAY) for x in (a, b))
    if np.abs(cv2.resize(ga, (32, 32)).astype(np.float32) - cv2.resize(gb, (32, 32)).astype(np.float32)).mean() >= mean_thresh:
        return True
    d = np.abs(cv2.resize(ga, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32)
               - cv2.resize(gb, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32))
    return float(np.percentile(d, 99)) > local_thresh

def sample_frames(frames, native_fps, fps, max_frames, side):
    """Same sampling as the app (neuroassist/ingest.py): step, keep last, drop unchanged, cap."""
    step = max(1, round(native_fps / fps))
    idx = [i for i in range(len(frames)) if i % step == 0]
    if idx[-1] != len(frames) - 1:
        idx.append(len(frames) - 1)
    kept = []
    for j, i in enumerate(idx):
        if kept and j != len(idx) - 1 and not frame_changed(frames[kept[-1]], frames[i]):
            continue
        kept.append(i)
    if len(kept) > max_frames:
        kept = [kept[s] for s in sorted(set(np.linspace(0, len(kept) - 1, max_frames).round().astype(int)))]
    out = []
    for i in kept:
        f = frames[i]; h, w = f.shape[:2]; s = side / max(h, w)
        if s < 1:
            f = cv2.resize(f, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
        out.append(Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
    return out, [i / native_fps for i in kept], len(frames) / native_fps

def to_jpeg(img, side):
    img = img.convert("RGB"); img.thumbnail((side, side))
    buf = io.BytesIO(); img.save(buf, "JPEG", quality=90); return buf.getvalue()

## 4. Synthetic video generator (preview below)

In [ ]:
# Synthetic clips with known answers. Our own generator, not the organizers' clips: random shapes,
# colors, sizes, speeds, backgrounds and wording, so the model learns the skills, not the clips.
W, H, NATIVE_FPS = 640, 360, 10
COLORS = {"red": (40, 40, 230), "blue": (220, 90, 30), "green": (60, 170, 40), "yellow": (30, 210, 240),
          "orange": (20, 140, 250), "purple": (170, 50, 140), "pink": (180, 120, 250), "black": (20, 20, 20)}
BGS = {"white": (245, 245, 245), "gray": (150, 150, 150), "light blue": (240, 220, 190), "beige": (200, 230, 240)}
SHAPES = ["circle", "square", "triangle"]
WORD_PAIRS = [("OPEN", "CLOSED"), ("PUSH", "PULL"), ("STOP", "GO"), ("IN", "OUT"), ("ON", "OFF"), ("YES", "NO"),
              ("ENTER", "EXIT"), ("HOT", "COLD"), ("UP", "DOWN"), ("SALE", "SOLD"), ("WALK", "WAIT"), ("FULL", "EMPTY")]

def canvas(bg):
    return np.full((H, W, 3), bg, np.uint8)

def draw(img, shape, cx, cy, r, color):
    cx, cy, r = int(cx), int(cy), int(r)
    if shape == "circle":
        cv2.circle(img, (cx, cy), r, color, -1, cv2.LINE_AA)
    elif shape == "square":
        cv2.rectangle(img, (cx - r, cy - r), (cx + r, cy + r), color, -1)
    else:
        pts = np.array([[cx, cy - r], [cx - r, cy + r], [cx + r, cy + r]], np.int32)
        cv2.fillPoly(img, [pts], color, cv2.LINE_AA)

def text(img, s, scale=3.0, color=(20, 20, 20)):
    (tw, th), _ = cv2.getTextSize(s, cv2.FONT_HERSHEY_SIMPLEX, scale, 6)
    cv2.putText(img, s, ((W - tw) // 2, (H + th) // 2), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 6, cv2.LINE_AA)

def plural(n, word):
    return word if n == 1 else word + "s"

def grid_positions(rng, n, r):
    cols = max(1, (W - 40) // (2 * r + 20)); rows = max(1, (H - 40) // (2 * r + 20))
    cells = rng.sample([(c, q) for c in range(cols) for q in range(rows)], n)
    return [(40 + r + c * (2 * r + 20), 40 + r + q * (2 * r + 20)) for c, q in cells]

def clip(rng):
    """Returns (frames_bgr, question, answer). Ten templates, varied wording."""
    t = rng.randrange(10)
    bg_name = rng.choice(list(BGS)); bg = BGS[bg_name]
    cols = [c for c in COLORS if not (bg_name == "white" and c == "yellow")]
    shape, color = rng.choice(SHAPES), rng.choice(cols)
    r = rng.randint(22, 40)
    frames = []
    if t == 0:                                            # shapes appear one at a time
        n = rng.randint(2, 9); pos = grid_positions(rng, n, r); per = rng.randint(4, 8)
        for k in range(n * per + per):
            img = canvas(bg)
            for i in range(min(n, k // per + 1) if k >= per else 0):
                draw(img, shape, *pos[i], r, COLORS[color])
            frames.append(img)
        q = rng.choice(["How many shapes appear in this video?", f"How many {shape}s appear?",
                        "Count the shapes that appear in the video."])
        a = f"{n} {plural(n, shape)} appear one at a time."
    elif t == 1:                                          # moves in one direction
        d = rng.choice(["left", "right", "up", "down"]); steps = rng.randint(20, 40)
        x0, y0 = (W * 0.75, H / 2) if d == "left" else (W * 0.25, H / 2) if d == "right" else \
                 (W / 2, H * 0.75) if d == "up" else (W / 2, H * 0.25)
        dx, dy = {"left": (-1, 0), "right": (1, 0), "up": (0, -1), "down": (0, 1)}[d]
        span = (W if dx else H) * 0.5
        for k in range(steps):
            img = canvas(bg); f = k / (steps - 1)
            draw(img, shape, x0 + dx * span * f, y0 + dy * span * f, r, COLORS[color]); frames.append(img)
        q = rng.choice([f"Which direction does the {shape} move?", f"Which way is the {color} {shape} moving?",
                        f"In which direction does the {shape} move?"])
        a = f"The {color} {shape} moves {d}."
    elif t == 2:                                          # three colors in order
        c3 = rng.sample(cols, 3); per = rng.randint(8, 14)
        for c in c3:
            for _ in range(per):
                img = canvas(bg); draw(img, shape, W / 2, H / 2, r * 2, COLORS[c]); frames.append(img)
        q = rng.choice(["What color appears first, and what color appears last?",
                        "Which color is shown first and which is shown last?"])
        a = f"A {c3[0]} {shape} appears first, then {c3[1]}, and {c3[2]} appears last."
    elif t == 3:                                          # sign text changes
        w1, w2 = rng.choice(WORD_PAIRS)
        if rng.random() < 0.5:
            w1, w2 = w2, w1
        per = rng.randint(10, 20)
        for w in (w1, w2):
            for _ in range(per):
                img = canvas(bg); cv2.rectangle(img, (120, 90), (W - 120, H - 90), (255, 255, 255), -1)
                cv2.rectangle(img, (120, 90), (W - 120, H - 90), (30, 30, 30), 4); text(img, w, 2.4); frames.append(img)
        q = rng.choice(["What does the sign say at the start and at the end?",
                        "What does the sign say first, and what does it say at the end?"])
        a = f'The sign says "{w1}" at the start and "{w2}" at the end.'
    elif t == 4:                                          # bar grows or shrinks
        grow = rng.random() < 0.5; steps = rng.randint(20, 40); lo, hi = rng.randint(20, 80), rng.randint(220, 320)
        for k in range(steps):
            f = k / (steps - 1); h = lo + (hi - lo) * (f if grow else 1 - f)
            img = canvas(bg); cv2.rectangle(img, (W // 2 - 40, int(H - 20 - h)), (W // 2 + 40, H - 20), COLORS[color], -1)
            frames.append(img)
        q = rng.choice(["Does the bar get taller or shorter?", "Is the bar getting taller or shorter?"])
        a = f"The bar gets {'taller' if grow else 'shorter'}."
    elif t == 5:                                          # two shapes meet / move apart
        meet = rng.random() < 0.6; steps = rng.randint(20, 35); c2 = rng.choice([c for c in cols if c != color])
        for k in range(steps):
            f = k / (steps - 1); gap = (W * 0.35) * ((1 - f) if meet else f) + (r if meet else r * 1.2)
            img = canvas(bg)
            draw(img, shape, W / 2 - gap, H / 2, r, COLORS[color]); draw(img, shape, W / 2 + gap, H / 2, r, COLORS[c2])
            frames.append(img)
        q = rng.choice(["What happens to the two shapes?", "Describe what the two shapes do."])
        a = ("The two shapes move toward each other until they touch." if meet
             else "The two shapes move away from each other.")
    elif t == 6:                                          # countdown / count up
        k = rng.randint(3, 9); down = rng.random() < 0.7; per = rng.randint(6, 10)
        seq = list(range(k, 0, -1)) if down else list(range(1, k + 1))
        for v in seq:
            for _ in range(per):
                img = canvas(bg); text(img, str(v), 5.0, COLORS[color]); frames.append(img)
        q = rng.choice(["What number is shown last?", "What is the last number shown?"])
        a = (f"A countdown from {k} to 1; the last number shown is 1." if down
             else f"The numbers count up from 1 to {k}; the last number shown is {k}.")
    elif t == 7:                                          # appear, then some disappear
        n = rng.randint(3, 8); m = rng.randint(1, n - 1); pos = grid_positions(rng, n, r); per = rng.randint(4, 7)
        shown = []
        for i in range(n):
            shown.append(i)
            for _ in range(per):
                img = canvas(bg); [draw(img, shape, *pos[j], r, COLORS[color]) for j in shown]; frames.append(img)
        for i in rng.sample(range(n), m):
            shown.remove(i)
            for _ in range(per):
                img = canvas(bg); [draw(img, shape, *pos[j], r, COLORS[color]) for j in shown]; frames.append(img)
        left = n - m
        q = rng.choice(["Several shapes appear and then some disappear. How many shapes are left at the end?",
                        "How many shapes are left at the end?"])
        a = (f"{n} {plural(n, shape)} appear, {m} {'disappears' if m == 1 else 'disappear'}, "
             f"so {left} {'is' if left == 1 else 'are'} left at the end.")
    elif t == 8:                                          # ball bounces k times
        k = rng.randint(1, 5); per = rng.randint(10, 14); floor = H - 30; top = rng.randint(60, 140)
        for b in range(k):
            for s in range(per):
                f = s / per; y = floor - r - (floor - r - top) * (1 - (2 * f - 1) ** 2)
                img = canvas(bg); cv2.line(img, (0, floor), (W, floor), (60, 60, 60), 3)
                draw(img, "circle", 80 + (W - 160) * (b * per + s) / (k * per), y, r, COLORS[color]); frames.append(img)
        q = rng.choice(["How many times does the ball bounce?", "How many bounces does the ball make?"])
        a = f"The ball bounces {k} {plural(k, 'time')}."
    else:                                                 # background color changes
        c1, c2 = rng.sample(list(COLORS), 2); per = rng.randint(10, 20)
        for c in (c1, c2):
            for _ in range(per):
                frames.append(canvas(COLORS[c]))
        if rng.random() < 0.5:
            q, a = "What color is the background at the end?", f"The background ends {c2}."
        else:
            q, a = "What color is the background at the start?", f"The background starts {c1}."
    return frames, q, a

def video_example(seed):
    rng = random.Random(seed)
    frames, q, a = clip(rng)
    task = app_task(q, ALLOWED_VIDEO)
    fps = VIDEO_FPS_DENSE if task in DENSE_TASKS else VIDEO_FPS_SPARSE
    imgs, times, dur = sample_frames(frames, NATIVE_FPS, fps, VIDEO_MAX_FRAMES, VIDEO_SIDE)
    return imgs, video_prompt(q, times, dur, task), json.dumps({"answer": a}), a

imgs, prompt, target, _ = video_example(1)
print(prompt, "\n→", target, f"\n({len(imgs)} frames, {imgs[0].size})")
imgs[0]

## 5. Load the training data

In [ ]:
import torch
from datasets import load_dataset

def cauldron(subset, n):
    """n (jpeg, prompt, target) examples from a Cauldron TRAIN subset, in the app's formats."""
    out = []
    ds = load_dataset("HuggingFaceM4/the_cauldron", subset, split="train", streaming=True).shuffle(seed=SEED, buffer_size=1000)
    for row in ds:
        if len(out) >= n:
            break
        if len(row["images"]) != 1:
            continue
        turn = random.choice(row["texts"])
        q = turn["user"].split("\n")[0].strip()          # drop Cauldron's "Give a very brief answer." line
        a = turn["assistant"].strip().rstrip(".")
        made = image_example(q, a)
        if made:
            out.append((to_jpeg(row["images"][0], IMAGE_SIDE),) + made + (a,))
    print(f"{subset:<8} {len(out)}")
    return out

scale = 0.02 if SMOKE_TEST else 1.0
image_items, image_eval = [], []
for subset, n in CAULDRON.items():
    rows = cauldron(subset, int(n * scale) + (EVAL_IMAGES // len(CAULDRON)))
    image_eval += rows[: EVAL_IMAGES // len(CAULDRON)]
    image_items += rows[EVAL_IMAGES // len(CAULDRON):]
video_seeds = list(range(int(SYNTH_VIDEOS * scale) + 10))
video_eval_seeds = list(range(10**6, 10**6 + EVAL_VIDEOS))      # different seeds = unseen clips

class Mixed(torch.utils.data.Dataset):
    """Built lazily: images stay as JPEG bytes, clips are rendered on demand."""
    def __init__(self):
        self.items = [("img", i) for i in range(len(image_items))] + [("vid", s) for s in video_seeds]
        random.Random(SEED).shuffle(self.items)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        kind, ref = self.items[i]
        if kind == "img":
            jpg, prompt, target, _ = image_items[ref]
            images = [Image.open(io.BytesIO(jpg)).convert("RGB")]
        else:
            images, prompt, target, _ = video_example(ref)
        content = [{"type": "image", "image": im} for im in images] + [{"type": "text", "text": prompt}]
        return {"messages": [{"role": "user", "content": content},
                             {"role": "assistant", "content": [{"type": "text", "text": target}]}]}

train_ds = Mixed()
print(f"training examples: {len(image_items)} images + {len(video_seeds)} clips = {len(train_ds)}")

## 6. Load the model and add LoRA adapters

In [ ]:
import os
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))
OUT_DIR = f"/content/{RUN_NAME}"
if SAVE_TO_DRIVE and not SMOKE_TEST:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = f"{DRIVE_DIR}/{RUN_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)

from unsloth import FastVisionModel, is_bf16_supported
model, tokenizer = FastVisionModel.from_pretrained(MODEL_NAME, load_in_4bit=True, use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model, finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none", random_state=SEED)

## 7. Score the base model (before)

In [ ]:
NUM_RE = re.compile(r"\b(\d+|zero|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)\b", re.I)
KEY_WORDS = set(COLORS) | {"left", "right", "up", "down", "taller", "shorter", "toward", "away", "touch"}

def ask(images, prompt, max_new_tokens=96):
    msgs = [{"role": "user", "content": [{"type": "image"}] * len(images) + [{"type": "text", "text": prompt}]}]
    text_in = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    enc = tokenizer(images=images, text=[text_in], add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

def field(out, key):
    try:
        return str(json.loads(out).get(key, ""))
    except Exception:
        return out

def video_correct(pred, ref):
    keys = [w for w in re.findall(r"[a-z]+|\d+", ref.lower()) if w in KEY_WORDS or w.isdigit()]
    keys += [w.lower() for w in re.findall(r'"([^"]+)"', ref)]
    p = pred.lower()
    if re.search(r"how many|what number|last number", ref.lower()) or not keys:
        return bool(NUM_RE.findall(ref)) and NUM_RE.findall(ref)[-1].lower() in [n.lower() for n in NUM_RE.findall(p)]
    return all(re.search(rf"\b{re.escape(k)}\b", p) for k in keys)

def evaluate_model():
    FastVisionModel.for_inference(model)
    img_ok = 0
    for jpg, prompt, target, ans in image_eval:
        out = ask([Image.open(io.BytesIO(jpg)).convert("RGB")], prompt)
        gold = json.loads(target)
        got = field(out, "count" if "count" in gold else "answer").strip().lower().rstrip(".")
        img_ok += got == str(gold.get("count", gold.get("answer"))).lower()
    vid_ok = 0
    for s in video_eval_seeds[: 20 if SMOKE_TEST else len(video_eval_seeds)]:
        images, prompt, _, ref = video_example(s)
        vid_ok += video_correct(field(ask(images, prompt), "answer"), ref)
    FastVisionModel.for_training(model)
    n_vid = 20 if SMOKE_TEST else len(video_eval_seeds)
    return {"image exact match": round(img_ok / len(image_eval), 3),
            "synthetic video correct": round(vid_ok / n_vid, 3)}

print("Scoring the base model before training...")
before = evaluate_model()
print(before)

## 8. Train, then score again

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

try:      # answers-only loss (newer Unsloth); falls back to full-sequence loss
    collator = UnslothVisionDataCollator(model, tokenizer, train_on_responses_only=True,
                                         instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
except TypeError:
    collator = UnslothVisionDataCollator(model, tokenizer)

wanted = dict(
    output_dir=f"/content/checkpoints/{RUN_NAME}",
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS, max_steps=10 if SMOKE_TEST else -1,
    learning_rate=LEARNING_RATE, warmup_ratio=0.03, lr_scheduler_type="cosine",
    optim="adamw_8bit", weight_decay=0.01, bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
    logging_steps=5 if SMOKE_TEST else 20, save_strategy="no", seed=SEED, report_to="none",
    remove_unused_columns=False, dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True},
    max_length=MAX_SEQ_LENGTH, max_seq_length=MAX_SEQ_LENGTH, dataloader_num_workers=2)
accepted = inspect.signature(SFTConfig.__init__).parameters
if "max_length" in accepted:
    wanted.pop("max_seq_length")
args = SFTConfig(**{k: v for k, v in wanted.items() if k in accepted})
kw = dict(model=model, train_dataset=train_ds, data_collator=collator, args=args)
kw["processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters
   else "tokenizer"] = tokenizer
FastVisionModel.for_training(model)
trainer = SFTTrainer(**kw)
stats = trainer.train()
print(f"done in {stats.metrics['train_runtime'] / 60:.1f} min · loss {stats.metrics['train_loss']:.3f}")

after = evaluate_model()
print(f"{'metric':<28}{'before':>8}{'after':>8}")
for k in before:
    print(f"{k:<28}{before[k]:>8.3f}{after[k]:>8.3f}")
json.dump({"model": MODEL_NAME, "before": before, "after": after}, open(f"{OUT_DIR}/scores.json", "w"), indent=2)

## 9. Save adapters and the merged model for Ollama

In [ ]:
model.save_pretrained(f"{OUT_DIR}/lora"); tokenizer.save_pretrained(f"{OUT_DIR}/lora")
merged = f"{OUT_DIR}/merged"
model.save_pretrained_merged(merged, tokenizer, save_method="merged_16bit")   # full weights, ~7.5 GB
open(f"{merged}/Modelfile", "w").write("FROM .\nPARAMETER temperature 0\n")
print("Merged model + Modelfile in", merged)
print(f"""
On the demo laptop, copy the whole 'merged' folder from Drive, then inside it:
    ollama create {OLLAMA_NAME} --quantize q4_K_M -f Modelfile
and run the app / eval with:
    $env:{APP_SETTING}="{OLLAMA_NAME}"
    $env:NEUROASSIST_COUNT_MODE="direct"        # this model was trained on direct counts
    $env:NEUROASSIST_VIDEO_MODE="direct"        # and on all frames in one call
    $env:NEUROASSIST_VIDEO_DIRECT_MAX_FRAMES="16"
""")

## 10. Check the export loads in Ollama (runs inside Colab)

In [ ]:
# Proves Ollama can import and run the exported model, using Ollama installed inside Colab.
# If this fails, do NOT spend an hour training: keep the stock qwen3-vl:4b in the app instead.
import subprocess, time, base64, urllib.request
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
r = subprocess.run(["ollama", "create", OLLAMA_NAME, "--quantize", "q4_K_M", "-f", "Modelfile"],
                   cwd=merged, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "ollama create failed: see the log above"
images, prompt, _, ref = video_example(10**6)
buf = io.BytesIO(); images[0].save(buf, "JPEG")
body = json.dumps({"model": OLLAMA_NAME, "stream": False, "format": "json",
                   "messages": [{"role": "user", "content": "What is in this image? Reply as JSON with key answer.",
                                 "images": [base64.b64encode(buf.getvalue()).decode()]}]}).encode()
reply = json.loads(urllib.request.urlopen(urllib.request.Request(
    "http://127.0.0.1:11434/api/chat", body, {"Content-Type": "application/json"}), timeout=300).read())
print("Ollama answered:", reply["message"]["content"])
print("\n✅ Export path works. Set SMOKE_TEST = False in Settings and Run all for the real training.")